In [0]:

# Notebook: 04_gold_layer.py
# Purpose : Create Gold views for reporting (patient summary, encounter timeline)

from pyspark.sql import SparkSession
from config.config import SILVER_DB, GOLD_DB
from importlib import reload
import sys
import requests, uuid
from datetime import datetime
spark = SparkSession.builder.getOrCreate()

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")


# Add repository root to Python path for module imports
repo_root = "/Workspace/Users/vyshnavi.thunuguntla@outlook.com/databricks-repo-FHIR"
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Import and reload to pick up latest changes
import audit_utils
reload(audit_utils)
from audit_utils import audit_start, audit_end

# ── 1. Patient Summary ────────────────────────────────────────────────────────
run_id = str(uuid.uuid4()) 
vw_patient_summary = f"{GOLD_DB}.vw_patient_summary"
CNT1 = spark.sql(f"SELECT COUNT(*) FROM {GOLD_DB}.vw_patient_summary ").collect()[0][0]
started_at = datetime.utcnow() # current_timestamp(),
audit_start(
        run_id        = run_id,
        stage         = "gold",
        resource_type = 'vw_patient_summary',
        operation     = "VIEW_REFRESH",
        status        = "STARTED",
        started_at = started_at ,
        before_run_source_count = CNT1 ,
        created_by    = "04_gold_layer"
    )

spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_DB}.vw_patient_summary AS
SELECT
    p.patient_id,
    p.family_name,
    p.given_name,
    p.gender,
    p.birth_date,
    COUNT(DISTINCT e.encounter_id)   AS total_encounters,
    COUNT(DISTINCT c.condition_id)   AS total_conditions,
    COUNT(DISTINCT o.observation_id) AS total_observations
FROM {SILVER_DB}.patient       p
LEFT JOIN {SILVER_DB}.encounter   e ON e.patient_id = p.patient_id
LEFT JOIN {SILVER_DB}.condition   c ON c.patient_id = p.patient_id
LEFT JOIN {SILVER_DB}.observation o ON o.patient_id = p.patient_id
GROUP BY 1,2,3,4,5
""")

#- Audit--end ----
CNT2 = spark.sql(f"SELECT COUNT(*) FROM {GOLD_DB}.vw_patient_summary ").collect()[0][0]
audit_end(
        run_id        = run_id,
        stage         = "GOLD",
        resource_type = 'vw_patient_summary',
        operation     = "VIEW_REFRESH",
        status        = "ENDED",
        post_run_source_count = CNT2 ,
    )
# ── 2. Encounter Timeline ─────────────────────────────────────────────────────
#-Audit Start -----------
run_id = str(uuid.uuid4()) 
vw_encounter_timeline = f"{GOLD_DB}.vw_encounter_timeline"
CNT1 = spark.sql(f"SELECT COUNT(*) FROM {GOLD_DB}.vw_encounter_timeline ").collect()[0][0]
started_at = datetime.utcnow() # current_timestamp(),
audit_start(
        run_id        = run_id,
        stage         = "GOLD",
        resource_type = 'vw_encounter_timeline',
        operation     = "VIEW_REFRESH",
        status        = "STARTED",
        started_at = started_at ,
        before_run_source_count = CNT1 ,
        created_by    = "04_gold_layer"
    )

spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_DB}.vw_encounter_timeline AS
SELECT
    e.encounter_id,
    e.patient_id,
    p.family_name,
    p.given_name,
    e.encounter_class,
    e.status,
    e.period_start,
    e.period_end,
    DATEDIFF(e.period_end, e.period_start) AS length_of_stay_days
FROM {SILVER_DB}.encounter e
JOIN {SILVER_DB}.patient   p ON p.patient_id = e.patient_id
""")
#- Audit--end ----
CNT2 = spark.sql(f"SELECT COUNT(*) FROM {GOLD_DB}.vw_patient_summary ").collect()[0][0]
audit_end(
        run_id        = run_id,
        stage         = "GOLD",
        resource_type = 'vw_patient_summary',
        operation     = "VIEW_REFRESH",
        status        = "ENDED",
        post_run_source_count = CNT2 ,
    )

# ── 3. Condition Prevalence ───────────────────────────────────────────────────
#-Audit Start -----------
run_id = str(uuid.uuid4()) 
vw_condition_prevalence = f"{GOLD_DB}.vw_condition_prevalence"
CNT1 = spark.sql(f"SELECT COUNT(*) FROM {GOLD_DB}.vw_condition_prevalence ").collect()[0][0]
started_at = datetime.utcnow() # current_timestamp(),
audit_start(
        run_id        = run_id,
        stage         = "GOLD",
        resource_type = 'vw_condition_prevalence',
        operation     = "VIEW_REFRESH",
        status        = "STARTED",
        started_at = started_at ,
        before_run_source_count = CNT1 ,
        created_by    = "04_gold_layer"
    )
spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_DB}.vw_condition_prevalence AS
SELECT
    condition_name,
    snomed_code,
    COUNT(DISTINCT patient_id) AS patient_count,
    COUNT(*)                   AS total_diagnoses
FROM {SILVER_DB}.condition
WHERE clinical_status = 'active'
GROUP BY 1, 2
ORDER BY patient_count DESC
""")
#- Audit--end ----
CNT2 = spark.sql(f"SELECT COUNT(*) FROM {GOLD_DB}.vw_condition_prevalence ").collect()[0][0]
audit_end(
        run_id        = run_id,
        stage         = "GOLD",
        resource_type = 'vw_condition_prevalence',
        operation     = "VIEW_REFRESH",
        status        = "ENDED",
        post_run_source_count = CNT2 ,
    )
print("✅ Gold layer views created.")
